该文件用于visdom.py中，豆包模型返回内容的获取格式测试

In [1]:
from pdf2image import convert_from_path
import base64
import os
from openai import OpenAI
from io import BytesIO

In [2]:
# Test Doubao Responses API with a PDF page converted to image

pdf_path = "/home/zechuan/m3docrag/contents/2024_Tencent_ESG.pdf"

# Convert the first page of the PDF to a PIL image
pages = convert_from_path(pdf_path)
if not pages:
    raise RuntimeError(f"No pages found in PDF: {pdf_path}")

first_page = pages[0]

# Encode image to base64 and build data URL
buf = BytesIO()
first_page.save(buf, format="JPEG")
img_b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
data_url = f"data:image/jpeg;base64,{img_b64}"

# Initialize Doubao client (API key must be in ARK_API_KEY env var)
api_key = os.getenv("ARK_API_KEY")
if not api_key:
    raise RuntimeError("Please set ARK_API_KEY environment variable for Doubao")

client = OpenAI(
    base_url="https://ark.cn-beijing.volces.com/api/v3",
    api_key=api_key,
)

prompt = "简单描述一下这张 ESG 报告页面的主要内容，用中文回答。"

response = client.responses.create(
    model="doubao-seed-1-6-flash-250828",
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_image",
                    "image_url": data_url,
                },
                {
                    "type": "input_text",
                    "text": prompt,
                },
            ],
        }
    ],
)

print("type(response):", type(response))

# Try to inspect key fields to understand the structure
public_attrs = [a for a in dir(response) if not a.startswith("_")]
print("public attributes:", public_attrs)

try:
    print("type(response.output):", type(response.output))
    print("len(response.output):", len(response.output))
    print("type(response.output[0]):", type(response.output[0]))
    print("type(response.output[0].content):", type(response.output[0].content))
    print("type(response.output[0].content[0]):", type(response.output[0].content[0]))
    print("text field:", response.output[0].content[0].text)
except Exception as e:
    print("Error when accessing response.output structure:", repr(e))
    print("Raw response:", response)

type(response): <class 'openai.types.responses.response.Response'>
public attributes: ['background', 'construct', 'copy', 'created_at', 'dict', 'error', 'from_orm', 'id', 'incomplete_details', 'instructions', 'json', 'max_output_tokens', 'max_tool_calls', 'metadata', 'model', 'model_computed_fields', 'model_config', 'model_construct', 'model_copy', 'model_dump', 'model_dump_json', 'model_extra', 'model_fields', 'model_fields_set', 'model_json_schema', 'model_parametrized_name', 'model_post_init', 'model_rebuild', 'model_validate', 'model_validate_json', 'model_validate_strings', 'object', 'output', 'output_text', 'parallel_tool_calls', 'parse_file', 'parse_obj', 'parse_raw', 'previous_response_id', 'prompt', 'prompt_cache_key', 'reasoning', 'safety_identifier', 'schema', 'schema_json', 'service_tier', 'status', 'temperature', 'text', 'to_dict', 'to_json', 'tool_choice', 'tools', 'top_logprobs', 'top_p', 'truncation', 'update_forward_refs', 'usage', 'user', 'validate']
type(response.out

In [ ]:
response.output[1].content[0].text

'这是腾讯2024年环境、社会及管治（ESG）报告的封面页，左上角标注“Tencent 腾讯”标识，主标题为“Environmental, Social and Governance Report 2024”（2024年环境、社会及管治报告），背景是一幅展现山峦、河流与草原的自然风景图，整体以“环境”主题的自然景观呼应ESG报告中“环境”维度的核心内容。'

In [18]:
text_response = client.chat.completions.create(
    model="doubao-1-5-lite-32k-250115",
    messages=[
                {"role": "user", "content": "中国的首都是哪里？"},
    ],
)
text_response

ChatCompletion(id='0217647507632013b0ae744b08fb0a823328e49b8859288d746ae', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='中国的首都是北京。\n\n北京是中国的政治中心、文化中心、国际交往中心和科技创新中心，具有重要的历史地位和现代意义，承载着国家的诸多重要职能和活动。 ', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1764750764, model='doubao-1-5-lite-32k-250115', object='chat.completion', service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=44, prompt_tokens=14, total_tokens=58, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=None), prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cached_tokens=0)))

In [ ]:
import os
import requests
import json
BASE_URL = "https://api.zhizengzeng.com/v1"
API_SECRET_KEY = "sk-zk2786ad95a30e31524f4cf78f81247fcec6627c6e304b10"
# credit_grants
def credit_grants():
    api_secret_key = API_SECRET_KEY;  # 智增增的secret_key
    url = BASE_URL+'/dashboard/billing/credit_grants'; # 余额查询url
    headers = {'Content-Type': 'application/json', 'Accept':'application/json',
               'Authorization': "Bearer "+api_secret_key}
    resp = requests.post(url, headers=headers)
    resp = resp.json(); 
    json_str = json.dumps(resp, ensure_ascii=False)
    print(json_str)

In [7]:
credit_grants()

{"code": 9, "msg": "缺少请求头Authorization或Authorization不对", "error": {"message": "缺少请求头Authorization或Authorization不对", "type": "invalid_request_error"}}
